1. Загрузите объекты из новостного датасета 20 newsgroups, относящиеся к категориям "космос" и "атеизм" (инструкция приведена выше).

In [16]:
from sklearn import datasets
import numpy as np
newsgroups = datasets.fetch_20newsgroups(subset='all', categories=['alt.atheism', 'sci.space'])
X = newsgroups.data
y = newsgroups.target

2. Вычислите TF-IDF-признаки для всех текстов.

In [17]:


from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer()
data_tfidf = tv.fit_transform(X)


3. Подберите минимальный лучший параметр C из множества [10^(-5), ..., 10^5] для SVM с линейным ядром (kernel='linear') при помощи кросс-валидации по 5 блокам. Укажите параметр random_state=241 и для SVM, и для KFold. В качестве меры качества используйте долю верных ответов (accuracy).

In [18]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

grid = {'C': np.power(10.0, np.arange(-5,6))}
cv = KFold(n_splits=5, shuffle=True , random_state=241)
clf = SVC(kernel='linear' , random_state=241)
gs = GridSearchCV(clf,grid,scoring='accuracy',cv=cv)
gs.fit(data_tfidf,y)

import pandas as pd
results = pd.DataFrame(gs.cv_results_)
best_score = results['mean_test_score'].max()
best_scores = results['mean_test_score'] == best_score
best_Cs = results.loc[best_scores, 'param_C']

C_best = best_Cs.min()               
print('Best C:', C_best)

Best C: 1.0


4. Обучите SVM по всей выборке с лучшим параметром C, найденным на предыдущем шаге.

In [19]:

clf_final = SVC(kernel='linear', C=C_best, random_state=241)
clf_final.fit(data_tfidf, y)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


5. Найдите 10 слов с наибольшим по модулю весом. Они являются ответом на это задание. Укажите их через запятую, в нижнем регистре, в лексикографическом порядке.

In [20]:
coef = clf_final.coef_.toarray().flatten()
feature_names = tv.get_feature_names_out()

words_coef = [(word, abs(weight)) for word, weight in zip(feature_names, coef)]
top_words = sorted(words_coef, key=lambda x: x[1], reverse=True)[:10]
top_word_list = sorted([word for word, _ in top_words])

answer = ', '.join(top_word_list)
print(answer)

atheism, atheists, bible, god, keith, moon, religion, sci, sky, space
